# Forecasting Financial Inclusion in Ethiopia — Interim (Task 1 + Task 2)

This notebook focuses on:
- **Task 1**: Data exploration + light enrichment (reproducible)
- **Task 2**: EDA (plots + insights)

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib

# Headless-safe backend for nbconvert execution
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# Robust project-dir detection without relying on __file__ (not defined in notebooks).
# We search from the current working directory upwards.

def find_project_dir(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for base in candidates:
        if (base / "data" / "raw").exists() and (base / "src").exists():
            return base
        if (base / "ethiopia-fi-forecast" / "data" / "raw").exists():
            return base / "ethiopia-fi-forecast"
    raise FileNotFoundError("Could not locate project dir containing data/raw")


def save_fig(path: Path) -> Path:
    # Avoid tight_layout/bbox_inches='tight' here: with Python 3.14 + Matplotlib,
    # tight bbox calculation can hit a recursion error in some environments.
    plt.savefig(path, dpi=150)
    plt.close()
    return path


PROJECT_DIR = find_project_dir(Path.cwd())
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIG_DIR = PROJECT_DIR / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RAW_UNIFIED_CSV = RAW_DIR / "ethiopia_fi_unified_data.csv"
RAW_IMPACT_CSV = RAW_DIR / "impact_links.csv"
RAW_REF_CSV = RAW_DIR / "reference_codes.csv"
PROCESSED_ENRICHED_CSV = PROCESSED_DIR / "enriched_unified_data.csv"

(PROJECT_DIR, RAW_UNIFIED_CSV.exists(), PROCESSED_ENRICHED_CSV.exists())

(PosixPath('/home/aln_lvr/Desktop/Courses/KAIM/B8W10/Forecasting-Financial-Inclusion-B8W10/ethiopia-fi-forecast'),
 True,
 True)

## Task 1 — Load datasets

In [2]:
raw_unified = pd.read_csv(RAW_UNIFIED_CSV)
raw_impact = pd.read_csv(RAW_IMPACT_CSV)
raw_ref = pd.read_csv(RAW_REF_CSV)

enriched = pd.read_csv(PROCESSED_ENRICHED_CSV)

# parse date columns
for df in (raw_unified, raw_impact, enriched):
    for col in ["observation_date", "period_start", "period_end", "collection_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

print('raw_unified', raw_unified.shape)
print('raw_impact ', raw_impact.shape)
print('raw_ref    ', raw_ref.shape)
print('enriched   ', enriched.shape)

raw_unified (43, 34)
raw_impact  (14, 35)
raw_ref     (71, 4)
enriched    (51, 34)


/tmp/ipykernel_403038/4143362467.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


In [3]:
raw_unified.head(3)

,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaT,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaT,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaT,NaN,NaN


## Task 1 — Quick data audit (schema, missingness, duplicates)

In [4]:
def audit(df: pd.DataFrame, name: str) -> pd.DataFrame:
    missing_pct = (df.isna().mean() * 100).round(1)
    out = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_%': missing_pct,
        'n_unique': df.nunique(dropna=True),
    }).sort_values(['missing_%', 'n_unique'], ascending=[False, True])
    print(f'\n{name}: rows={len(df)}, cols={df.shape[1]}')
    print('duplicates (full row):', df.duplicated().sum())
    return out

audit_enriched = audit(enriched, 'enriched')
audit_enriched.head(15)


enriched: rows=51, cols=34
duplicates (full row): 0


,dtype,missing_%,n_unique
region,float64,100.0,0
related_indicator,float64,100.0,0
relationship_type,float64,100.0,0
impact_direction,float64,100.0,0
impact_magnitude,float64,100.0,0
impact_estimate,float64,100.0,0
lag_months,float64,100.0,0
evidence_basis,float64,100.0,0
collection_date,datetime64[ns],84.3,1
notes,object,84.3,1


In [5]:
# Visualize missingness for the enriched dataset (top 15 missing columns)
missing = (enriched.isna().mean() * 100).sort_values(ascending=False)
top_missing = missing.head(15)[::-1]

plt.figure(figsize=(8, 5))
plt.barh(top_missing.index.astype(str), top_missing.values)
plt.title('Top missing columns (enriched)')
plt.xlabel('Missing (%)')

out = FIG_DIR / 'missingness_top15.png'
save_fig(out)

PosixPath('/home/aln_lvr/Desktop/Courses/KAIM/B8W10/Forecasting-Financial-Inclusion-B8W10/ethiopia-fi-forecast/reports/figures/missingness_top15.png')

### What enrichment was added?

A small, reproducible enrichment is generated by running:

`python src/build_dataset.py`

It appends a yearly series (2014–2023) for **mobile cellular subscriptions per 100 people** (World Bank WDI indicator `IT.CEL.SETS.P2`) into the unified dataset under `indicator_code = ACC_MOBILE_PEN`.

Details are logged in `data_enrichment_log.md`.

## Task 2 — EDA

In [6]:
# Separate observations and events from the enriched dataset
obs = enriched[enriched['record_type'] == 'observation'].copy()
evts = enriched[enriched['record_type'] == 'event'].copy()

print('observations:', obs.shape)
print('events:', evts.shape)
print('obs date range:', obs['observation_date'].min(), '->', obs['observation_date'].max())

observations: (38, 34)
events: (10, 34)
obs date range: 2014-12-31 00:00:00 -> 2025-12-31 00:00:00


In [7]:
def get_series(df: pd.DataFrame, indicator_code: str) -> pd.DataFrame:
    out = df[df['indicator_code'] == indicator_code].copy()
    out = out.dropna(subset=['observation_date', 'value_numeric'])
    out = out.sort_values('observation_date')
    return out[['observation_date', 'value_numeric', 'unit', 'source_name', 'gender', 'location']]

series_codes = {
    'ACC_OWNERSHIP': 'Account Ownership Rate (%)',
    'ACC_MM_ACCOUNT': 'Mobile Money Account Rate (%)',
    'ACC_MOBILE_PEN': 'Mobile Subscription Penetration (%; per 100 people)',
    'USG_P2P_COUNT': 'P2P Transaction Count',
    'USG_TELEBIRR_USERS': 'Telebirr Registered Users',
}

{k: len(get_series(obs, k)) for k in series_codes}

{'ACC_OWNERSHIP': 6,
 'ACC_MM_ACCOUNT': 2,
 'ACC_MOBILE_PEN': 9,
 'USG_P2P_COUNT': 2,
 'USG_TELEBIRR_USERS': 1}

In [8]:
# Plot key time series (one figure per indicator)
for code, title in series_codes.items():
    s = get_series(obs, code)
    if s.empty:
        continue

    plt.figure(figsize=(8, 4))
    plt.plot(s['observation_date'], s['value_numeric'], marker='o')
    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel(s['unit'].dropna().iloc[0] if s['unit'].notna().any() else '')

    fname = f"ts_{code.lower()}.png"
    out = FIG_DIR / fname
    save_fig(out)
    print('saved', out.name)

saved ts_acc_ownership.png


saved ts_acc_mm_account.png


saved ts_acc_mobile_pen.png


saved ts_usg_p2p_count.png


saved ts_usg_telebirr_users.png


### Event overlay example (Account Ownership)

In [9]:
acc = enriched[(enriched['record_type'] == 'observation') & (enriched['indicator_code'] == 'ACC_OWNERSHIP') & (enriched['gender'] == 'all')].copy()
acc = acc.dropna(subset=['observation_date', 'value_numeric']).sort_values('observation_date')

ev = evts.dropna(subset=['observation_date']).sort_values('observation_date')

plt.figure(figsize=(10, 4))
plt.plot(acc['observation_date'], acc['value_numeric'], marker='o', label='Account ownership (all)')

for _, r in ev.iterrows():
    plt.axvline(r['observation_date'], color='gray', alpha=0.25)

plt.title('Account Ownership Rate with policy/event markers')
plt.xlabel('Date')
plt.ylabel('%')
plt.legend(loc='best')

out = FIG_DIR / 'account_ownership_with_events.png'
save_fig(out)

PosixPath('/home/aln_lvr/Desktop/Courses/KAIM/B8W10/Forecasting-Financial-Inclusion-B8W10/ethiopia-fi-forecast/reports/figures/account_ownership_with_events.png')

### Correlation snapshot (yearly pivot)

This is a simple correlation using a yearly pivot. With sparse data, treat it as **directional only**.

In [10]:
selected = ['ACC_OWNERSHIP', 'ACC_MM_ACCOUNT', 'ACC_MOBILE_PEN', 'USG_P2P_COUNT']
tmp = obs[obs['indicator_code'].isin(selected)].copy()
tmp = tmp.dropna(subset=['observation_date', 'value_numeric'])
tmp['year'] = tmp['observation_date'].dt.year

pivot = tmp.pivot_table(index='year', columns='indicator_code', values='value_numeric', aggfunc='mean')
corr = pivot.corr(min_periods=2)

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation (yearly mean)')

out = FIG_DIR / 'correlation_heatmap.png'
save_fig(out)

PosixPath('/home/aln_lvr/Desktop/Courses/KAIM/B8W10/Forecasting-Financial-Inclusion-B8W10/ethiopia-fi-forecast/reports/figures/correlation_heatmap.png')

## Key insights (based on the provided data)

1. **Account ownership increased materially** between 2014 → 2021 in the Global Findex snapshots, suggesting steady progress in formal financial access over the decade.
2. **Gender gap is visible** in the 2021 Findex snapshot (male vs female account ownership), indicating inclusion is improving but not evenly distributed.
3. **Mobile money account ownership is still low compared to overall account ownership**, but the direction is upward — suggesting digitization is happening, but not yet the dominant channel.
4. **Mobile subscription penetration (WDI enrichment) trends upward over time**, supporting the idea that mobile infrastructure can be an important enabler of digital financial inclusion.
5. **Digital payment usage is growing** (e.g., P2P counts), which aligns with broader shifts toward electronic channels and interoperability efforts.

## Key limitations

- **Sparse time series** for most indicators (many have only 1–2 observations), which limits forecasting quality.
- **Mixed definitions and sources** (survey snapshots vs admin reports vs per-100 subscriptions) mean indicators are not perfectly comparable.
- **Some fields are missing for many records** (e.g., region, relationship fields), limiting segmentation and causal linkage.
- **Events are included but impacts are not fully quantified** across all indicators; impact links are partial.
- Some records appear to be **targets or forward-looking placeholders**, so they should be separated from historical training data during modeling.